In [ ]:
GLCM extraction

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import cv2
import tifffile
# Compatible import for older and newer skimage versions
try:
    from skimage.feature.texture import graycomatrix, graycoprops
except ImportError:
    from skimage.feature import greycomatrix as graycomatrix, greycoprops as graycoprops
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# -----------------------------
# Configuration
# -----------------------------
data_dirs = {
    "Control": r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\Control",
    "CIS":     r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\CIS",
    "HIS":     r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\HIS"
}

# Output folder path
output_dir = r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\GLCM_features_280"
os.makedirs(output_dir, exist_ok=True)
output_csv_path = os.path.join(output_dir, "glcm_features_named.csv")

resize_size = 128
num_threads = 8  # Can be adjusted based on CPU core count

# Parameter lists
distances = [1, 2, 4, 8, 16, 32, 64]
angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
# Map angle radians to string labels for better readability
angle_labels = {0: "0", np.pi/4: "45", np.pi/2: "90", 3*np.pi/4: "135"}
properties = ['ASM', 'contrast', 'correlation', 'homogeneity', 'dissimilarity']

# -----------------------------
# Dynamically generate descriptive feature column names
# -----------------------------
def generate_feature_names(channel_name):
    """
    Generate column names following the flattening order of skimage.feature.greycoprops().flatten()
    Order: outer loop properties -> inner flatten (first distances, then angles)
    """
    feat_names = []
    for prop in properties:
        for d in distances:
            for a in angles:
                theta = angle_labels[a]
                # E.g.: ASM_d=1_theta=45_P
                name = f"{prop}_d={d}_theta={theta}_{channel_name}"
                feat_names.append(name)
    return feat_names

# Generate 140-dimensional feature names for P and S channels respectively, totaling 280 dimensions
p_feature_columns = generate_feature_names("P")
s_feature_columns = generate_feature_names("S")
columns = ['Label', 'P_Path', 'S_Path'] + p_feature_columns + s_feature_columns

# -----------------------------
# Helper Functions
# -----------------------------
def get_image_pairs(folder):
    """Match P/S image pairs"""
    p_files = sorted(glob.glob(os.path.join(folder, "*-P*.tif*")))
    pairs = []
    for p_file in p_files:
        s_file = p_file.replace("-P", "-S")
        if os.path.exists(s_file):
            pairs.append((p_file, s_file))
    return pairs

def extract_glcm_features(img):
    """Extract GLCM features for a single image (after normalization)"""
    img_uint8 = (img * 255).astype(np.uint8)
    # Use graycomatrix to avoid deprecation warnings in skimage 1.0+
    glcm = graycomatrix(img_uint8, distances=distances, angles=angles, symmetric=True, normed=True)
    feats = []
    for prop in properties:
        # graycoprops returns a 2D array of shape (len(distances), len(angles))
        # .flatten() flattens row-major (i.e., fix d, iterate through all theta, then next d)
        feats.extend(graycoprops(glcm, prop).flatten())
    return feats  # 140-dimensional

def process_pair(pair):
    p_file, s_file, label = pair
    try:
        p_img = cv2.resize(tifffile.imread(p_file).astype(np.float32)/65535., (resize_size, resize_size))
        s_img = cv2.resize(tifffile.imread(s_file).astype(np.float32)/65535., (resize_size, resize_size))
        p_feats = extract_glcm_features(p_img)
        s_feats = extract_glcm_features(s_img)
        return [label, p_file, s_file] + p_feats + s_feats
    except Exception as e:
        print(f"Error processing {p_file} / {s_file}: {e}")
        return None

# -----------------------------
# Build list of all image pairs
# -----------------------------
all_pairs = []
for label, folder in data_dirs.items():
    pairs = get_image_pairs(folder)
    print(f"Found {len(pairs)} image pairs in folder [{label}]")
    all_pairs.extend([(p, s, label) for p, s in pairs])

# -----------------------------
# Parallel Processing
# -----------------------------
results = []
with ThreadPoolExecutor(max_workers=num_threads) as executor:
    futures = [executor.submit(process_pair, pair) for pair in all_pairs]
    for f in tqdm(as_completed(futures), total=len(futures), desc="Processing all pairs"):
        res = f.result()
        if res is not None:
            results.append(res)

# -----------------------------
# Save to CSV
# -----------------------------
df = pd.DataFrame(results, columns=columns)
df.to_csv(output_csv_path, index=False)
print(f"GLCM features successfully saved to: {output_csv_path}")

In [ ]:
SVM and RF classification

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# -----------------------------
# Configuration
# -----------------------------
csv_path = r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\GLCM_features_280\glcm_features_named.csv"
output_dir = r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\GLCM_features_280"
os.makedirs(output_dir, exist_ok=True)
summary_csv_path = os.path.join(output_dir, "classifier_comparison_summary.csv")

# Random seeds as requested
seeds = [42, 100, 2026]

# Define the 5 classifiers to screen
classifiers = {
    "SVM-Linear": SVC(kernel='linear', random_state=42),
    "SVM-RBF": SVC(kernel='rbf', random_state=42),
    "SVM-Poly": SVC(kernel='poly', degree=3, random_state=42),
    "SVM-Sigmoid": SVC(kernel='sigmoid', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

# -----------------------------
# Load Data
# -----------------------------
print(f"Loading dataset from: {csv_path}")
df = pd.read_csv(csv_path)

# Separate features and labels
# Metadata columns: 'Label', 'P_Path', 'S_Path'
X = df.drop(columns=['Label', 'P_Path', 'S_Path']).values
y = df['Label'].values

print(f"Dataset shape: {X.shape[0]} samples, {X.shape[1]} features.")
print(f"Class distribution:\n{pd.Series(y).value_counts()}")

# -----------------------------
# Evaluation Pipeline
# -----------------------------
all_results = []

for seed in seeds:
    print(f"\n--- Running with Random Seed: {seed} ---")
    
    # 1. Split into Train (80%), Val (10%), Test (10%)
    # First split: Train+Val (90%) vs Test (10%)
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.10, random_state=seed, stratify=y
    )
    # Second split: Train (88.8% of train_val -> ~80% total) vs Val (11.1% of train_val -> ~10% total)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=1/9, random_state=seed, stratify=y_train_val
    )
    
    # Feature scaling (important for SVM)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # 2. 3-Fold Cross-Validation on the Training set
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    
    for clf_name, clf in classifiers.items():
        fold_accuracies = []
        
        for train_idx, val_idx in skf.split(X_train_scaled, y_train):
            X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
            y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]
            
            clf.fit(X_fold_train, y_fold_train)
            preds = clf.predict(X_fold_val)
            acc = accuracy_score(y_fold_val, preds)
            fold_accuracies.append(acc)
            
        mean_cv_acc = np.mean(fold_accuracies)
        std_cv_acc = np.std(fold_accuracies)
        
        # Train on full X_train and evaluate on Validation set
        clf.fit(X_train_scaled, y_train)
        val_acc = accuracy_score(y_val, clf.predict(X_val_scaled))
        
        # Test accuracy (optional generalizability check)
        test_acc = accuracy_score(y_test, clf.predict(X_test_scaled))
        
        print(f"[{clf_name}] CV Acc: {mean_cv_acc:.4f} (±{std_cv_acc:.4f}) | Val Acc: {val_acc:.4f} | Test Acc: {test_acc:.4f}")
        
        all_results.append({
            "Classifier": clf_name,
            "Seed": seed,
            "CV_Mean_Accuracy": mean_cv_acc,
            "CV_SD": std_cv_acc,
            "Validation_Accuracy": val_acc,
            "Test_Accuracy": test_acc
        })

# -----------------------------
# Aggregate and Save Summary
# -----------------------------
results_df = pd.DataFrame(all_results)

# Group by classifier to get overall mean & SD across the 3 random seeds
summary = results_df.groupby("Classifier").agg({
    "CV_Mean_Accuracy": ["mean", "std"],
    "Validation_Accuracy": ["mean", "std"],
    "Test_Accuracy": ["mean", "std"]
}).reset_index()

# Flatten column multi-index
summary.columns = ['_'.join(col).strip('_') for col in summary.columns.values]
summary = summary.sort_values(by="CV_Mean_Accuracy_mean", ascending=False)

print("\n==========================================")
print("Classifier Screening Summary (Across Seeds)")
print("==========================================")
print(summary.to_string(index=False))

summary.to_csv(summary_csv_path, index=False)
print(f"\nSummary results saved to: {summary_csv_path}")

In [ ]:
Search for C value

In [ ]:
import os
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# -----------------------------
# 1. Configuration of Paths and Parameters
# -----------------------------
csv_path = r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\GLCM_features_280\glcm_features_named.csv"
output_report_dir = r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\GLCM_features_280\GridSearch_Report"
os.makedirs(output_report_dir, exist_ok=True)

random_seed = 42

print(f"Loading dual-channel GLCM feature file: {csv_path} ...")
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"File not found, please check the path: {csv_path}")

df = pd.read_csv(csv_path)

# Extract features and labels (excluding metadata columns 'Label', 'P_Path', 'S_Path')
metadata_cols = ['Label', 'P_Path', 'S_Path']
feature_cols = [col for col in df.columns if col not in metadata_cols]
X = df[feature_cols].values
y = df['Label'].values

# Grid search settings
C_values = [0.001, 0.1, 1, 5, 10, 50, 100, 150, 200]
seeds = [42, 123, 2026]  # 3 different random seeds
n_splits = 5             # 5-fold cross-validation

# -----------------------------
# 2. Execute SVM-linear Grid Search
# -----------------------------
print(f"\n==============================================")
print(f"Starting SVM-linear grid search for dual-channel features (280-dim)...")
print(f"==============================================")

results = []
start_time = time.time()

# Use tqdm to display the grid search progress bar for C values
for C in tqdm(C_values, desc="Grid Searching C values"):
    fold_accuracies = []
    
    # Iterate through 3 random seeds
    for seed in seeds:
        # 1. 8:1:1 split (first split out 10% test set)
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X, y, test_size=0.1, random_state=seed, stratify=y
        )
        
        # 2. 5-fold cross-validation (strict fold-wise independent normalization and evaluation on 90% train_val set)
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for train_idx, val_idx in skf.split(X_train_val, y_train_val):
            X_tr, X_v = X_train_val[train_idx], X_train_val[val_idx]
            y_tr, y_v = y_train_val[train_idx], y_train_val[val_idx]
            
            # Strict anti-leakage: fit the scaler using only the current fold's training set
            scaler = RobustScaler()
            X_tr_scaled = scaler.fit_transform(X_tr)
            X_v_scaled = scaler.transform(X_v)
            
            # Use LinearSVC to speed up linear kernel computation
            clf = LinearSVC(C=C, random_state=seed, dual="auto", max_iter=10000)
            clf.fit(X_tr_scaled, y_tr)
            
            acc = accuracy_score(y_v, clf.predict(X_v_scaled))
            fold_accuracies.append(acc * 100)  # Convert to percentage
            
    mean_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    
    results.append({
        "C_Value": C,
        "Mean_Accuracy(%)": round(mean_acc, 2),
        "Std_Dev(%)": round(std_acc, 2)
    })
    print(f"\n  👉 C = {C:<6} | Mean Acc: {mean_acc:.2f}%  ±  SD: {std_acc:.2f}%")

elapsed_time = time.time() - start_time
print(f"\n⏱️ Total grid search elapsed time: {elapsed_time:.2f} seconds")

# -----------------------------
# 3. Save Grid Search Report
# -----------------------------
df_res = pd.DataFrame(results)
out_csv = os.path.join(output_report_dir, "DualChannel_svm_linear_grid_search.csv")
df_res.to_csv(out_csv, index=False)

# Print summary table in the console
print(f"\n📊 Dual-Channel Features Grid Search Summary Table:")
print(df_res.to_string(index=False))
print(f"\n✅ Dual-channel features grid search results successfully saved to: {out_csv}")

# Print the optimal C value
best_row = df_res.loc[df_res['Mean_Accuracy(%)'].idxmax()]
print(f"🏆 Optimal Dual-Channel Parameters: C = {best_row['C_Value']} (Accuracy: {best_row['Mean_Accuracy(%)']}% ± {best_row['Std_Dev(%)']}%)")
print("\n🎉 All grid search and evaluation processes completed successfully!")

In [ ]:
Performance of C=5

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score, 
    precision_recall_fscore_support
)

# -----------------------------
# 1. Configuration & Setup
# -----------------------------
csv_path = r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\GLCM_features_280\glcm_features_named.csv"
output_dir = r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\GLCM_features_280\Evaluation_C5"
os.makedirs(output_dir, exist_ok=True)

C_optimal = 5
seeds = [42, 123, 2026]
n_splits = 3  # 3-fold cross-validation

print(f"Loading dataset from: {csv_path} ...")
df = pd.read_csv(csv_path)

metadata_cols = ['Label', 'P_Path', 'S_Path']
feature_cols = [col for col in df.columns if col not in metadata_cols]
X = df[feature_cols].values
y = df['Label'].values
classes = np.unique(y)

print(f"Classes found: {classes}")

# -----------------------------
# 2. Cross-Validation & Evaluation Loop (8:1:1 split structure)
# -----------------------------
all_y_true = []
all_y_pred = []

print(f"\nRunning 3-seed x 3-fold CV evaluation with C={C_optimal}...")

for seed in seeds:
    # 8:1:1 split: 90% train_val, 10% test (mimicking your pipeline structure)
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.1, random_state=seed, stratify=y
    )
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for train_idx, val_idx in skf.split(X_train_val, y_train_val):
        X_tr, X_v = X_train_val[train_idx], X_train_val[val_idx]
        y_tr, y_v = y_train_val[train_idx], y_train_val[val_idx]
        
        # Scaling
        scaler = RobustScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_v_scaled = scaler.transform(X_v)
        
        # Train LinearSVC
        clf = LinearSVC(C=C_optimal, random_state=seed, dual="auto", max_iter=10000)
        clf.fit(X_tr_scaled, y_tr)
        
        preds = clf.predict(X_v_scaled)
        
        all_y_true.extend(y_v)
        all_y_pred.extend(preds)

all_y_true = np.array(all_y_true)
all_y_pred = np.array(all_y_pred)

# -----------------------------
# 3. Compute Metrics (4 Decimal Places, Raw Floats)
# -----------------------------
overall_accuracy = accuracy_score(all_y_true, all_y_pred)
report_dict = classification_report(all_y_true, all_y_pred, target_names=classes, output_dict=True)

print("\n==============================================")
print(f" PERFORMANCE METRICS (C = {C_optimal})")
print("==============================================")
print(f"Overall Accuracy: {overall_accuracy:.4f}\n")

print(f"{'Class':<12} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10}")
print("-" * 50)
for cls in classes:
    p = report_dict[cls]['precision']
    r = report_dict[cls]['recall']
    f = report_dict[cls]['f1-score']
    print(f"{cls:<12} | {p:<10.4f} | {r:<10.4f} | {f:<10.4f}")

print("-" * 50)
print(f"{'Macro Avg':<12} | {report_dict['macro avg']['precision']:<10.4f} | {report_dict['macro avg']['recall']:<10.4f} | {report_dict['macro avg']['f1-score']:<10.4f}")
print(f"{'Weighted Avg':<12} | {report_dict['weighted avg']['precision']:<10.4f} | {report_dict['weighted avg']['recall']:<10.4f} | {report_dict['weighted avg']['f1-score']:<10.4f}")

# Save metrics report to CSV
metrics_df = pd.DataFrame(report_dict).transpose()
metrics_df.to_csv(os.path.join(output_dir, "classification_report_C5.csv"))

# -----------------------------
# 4. Confusion Matrix Plot
# -----------------------------
cm = confusion_matrix(all_y_true, all_y_pred, labels=classes)

plt.figure(figsize=(7, 6), dpi=300)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, cbar=True)
plt.title(f'Confusion Matrix (SVM-Linear, C={C_optimal})', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=11)
plt.ylabel('True Label', fontsize=11)
plt.tight_layout()
cm_path = os.path.join(output_dir, "confusion_matrix.png")
plt.savefig(cm_path, dpi=300)
plt.close()
print(f"\nConfusion matrix saved to: {cm_path}")

# -----------------------------
# 5. One-vs-Rest (OvR) Pairwise Accuracy Matrix
# -----------------------------
# Computes pairwise accuracy for each binary combination (e.g., Control vs CIS)
n_classes = len(classes)
pairwise_acc_matrix = np.zeros((n_classes, n_classes))

for i in range(n_classes):
    for j in range(n_classes):
        if i == j:
            pairwise_acc_matrix[i, j] = 1.0  # Perfect self-accuracy
        else:
            # Filter subset containing only class i and class j
            mask = (all_y_true == classes[i]) | (all_y_true == classes[j])
            sub_true = all_y_true[mask]
            sub_pred = all_y_pred[mask]
            
            # Recalculate accuracy strictly between these two classes
            if len(sub_true) > 0:
                pairwise_acc_matrix[i, j] = accuracy_score(sub_true, sub_pred)

pairwise_df = pd.DataFrame(pairwise_acc_matrix, index=classes, columns=classes)
print("\n--- One-vs-Rest / Pairwise Accuracy Matrix ---")
print(pairwise_df.round(4).to_string())
pairwise_df.to_csv(os.path.join(output_dir, "pairwise_accuracy_matrix.csv"))

# Plot Pairwise Accuracy Matrix
plt.figure(figsize=(7, 6), dpi=300)
sns.heatmap(pairwise_df, annot=True, fmt='.4f', cmap='Purples', vmin=0, vmax=1, xticklabels=classes, yticklabels=classes, cbar=True)
plt.title(f'Pairwise Class Accuracy Matrix (C={C_optimal})', fontsize=12, fontweight='bold')
plt.xlabel('Target Class', fontsize=11)
plt.ylabel('Reference Class', fontsize=11)
plt.tight_layout()
pairwise_path = os.path.join(output_dir, "pairwise_accuracy_matrix.png")
plt.savefig(pairwise_path, dpi=300)
plt.close()
print(f"Pairwise accuracy matrix plot saved to: {pairwise_path}")
print("\nAll evaluation reports and plots generated successfully!")

In [ ]:
SHAP analysis

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from tqdm import tqdm
from joblib import Parallel, delayed
import contextlib
import joblib

# -----------------------------
# 1. Configuration of Paths and Parameters (Reduced training scale for speedup)
# -----------------------------
csv_path = r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\GLCM_features_280\glcm_features_named.csv"
output_report_dir = r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\GLCM_features_280\SHAP_Report_Light"
os.makedirs(output_report_dir, exist_ok=True)

C_value = 5              # Optimal parameter C=5
seeds = [42]             # [Speedup Optimization] Use 1 seed for debugging first, change to [42, 123, 2026] after verification
n_splits = 5             # 5-fold cross-validation
n_jobs = -1              # Use all CPU cores for parallel acceleration

print(f"Loading dual-channel GLCM feature file: {csv_path} ...")
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"File not found, please check the path: {csv_path}")

df = pd.read_csv(csv_path)

metadata_cols = ['Label', 'P_Path', 'S_Path']
feature_cols = [col for col in df.columns if col not in metadata_cols]
X = df[feature_cols].values
y = df['Label'].values
feature_names = np.array(feature_cols)

# -----------------------------
# 2. Compute SHAP Feature Importance
# -----------------------------
print(f"\n==============================================")
print(f"Starting SHAP feature importance calculation (C = {C_value}) ...")
print(f"==============================================")

global_shap_importances = np.zeros(X.shape[1])
total_runs = 0

for seed in seeds:
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.1, random_state=seed, stratify=y
    )
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    for train_idx, val_idx in tqdm(skf.split(X_train_val, y_train_val), desc=f"Seed {seed} SHAP"):
        X_tr, X_v = X_train_val[train_idx], X_train_val[val_idx]
        y_tr, y_v = y_train_val[train_idx], y_train_val[val_idx]
        
        scaler = RobustScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_v_scaled = scaler.transform(X_v)
        
        clf = LinearSVC(C=C_value, random_state=seed, dual="auto", max_iter=10000)
        clf.fit(X_tr_scaled, y_tr)
        
        explainer = shap.LinearExplainer(clf, X_tr_scaled)
        shap_values = explainer(X_v_scaled)
        
        vals = np.abs(shap_values.values if hasattr(shap_values, 'values') else shap_values)
        if vals.ndim == 3:
            vals = vals.mean(axis=2)
            
        fold_importance = vals.mean(axis=0)
        global_shap_importances += fold_importance
        total_runs += 1

global_shap_importances /= total_runs

shap_df = pd.DataFrame({
    'Feature': feature_names,
    'SHAP_Importance': global_shap_importances
}).sort_values(by='SHAP_Importance', ascending=False).reset_index(drop=True)

shap_csv_path = os.path.join(output_report_dir, f"SHAP_feature_importance_C{C_value}.csv")
shap_df.to_csv(shap_csv_path, index=False)
print(f"✅ SHAP feature importance successfully saved to: {shap_csv_path}")

# -----------------------------
# 3. Parallel Evaluation of Cumulative Feature Selection Performance (Increased step size)
# -----------------------------
print(f"\n==============================================")
print(f"Starting parallel evaluation of cumulative feature selection (Step size = 20) ...")
print(f"==============================================")

# [Speedup Optimization] Change step size from 10 to 20 to significantly reduce evaluation points
feature_counts = [1] + list(range(20, len(feature_cols) + 1, 20))
if feature_counts[-1] != len(feature_cols):
    feature_counts.append(len(feature_cols))

sorted_features = shap_df['Feature'].values
feature_name_to_idx = {name: idx for idx, name in enumerate(feature_cols)}

def evaluate_single_subset(k):
    top_k_names = sorted_features[:k]
    top_k_indices = [feature_name_to_idx[name] for name in top_k_names]
    
    subset_fold_accuracies = []
    for seed in seeds:
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X, y, test_size=0.1, random_state=seed, stratify=y
        )
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        
        for train_idx, val_idx in skf.split(X_train_val, y_train_val):
            X_tr, X_v = X_train_val[train_idx][:, top_k_indices], X_train_val[val_idx][:, top_k_indices]
            y_tr, y_v = y_train_val[train_idx], y_train_val[val_idx]
            
            scaler = RobustScaler()
            X_tr_scaled = scaler.fit_transform(X_tr)
            X_v_scaled = scaler.transform(X_v)
            
            clf = LinearSVC(C=C_value, random_state=seed, dual="auto", max_iter=10000)
            clf.fit(X_tr_scaled, y_tr)
            
            acc = accuracy_score(y_v, clf.predict(X_v_scaled))
            subset_fold_accuracies.append(acc * 100)
            
    mean_acc = np.mean(subset_fold_accuracies)
    std_acc = np.std(subset_fold_accuracies) if len(subset_fold_accuracies) > 1 else 0.0
    
    return {
        "Num_Features": k,
        "Mean_Accuracy(%)": round(mean_acc, 2),
        "Std_Dev(%)": round(std_acc, 2)
    }

# Parallel computation wrapper with real-time progress bar
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)
    old_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_callback
        tqdm_object.close()

start_time = time.time()
with tqdm_joblib(tqdm(total=len(feature_counts), desc="Evaluating Feature Subsets")):
    cumulative_results = Parallel(n_jobs=n_jobs)(
        delayed(evaluate_single_subset)(k) for k in feature_counts
    )

print(f"\n⏱️ Total cumulative feature evaluation elapsed time: {time.time() - start_time:.2f} seconds")

df_cum = pd.DataFrame(cumulative_results).sort_values(by="Num_Features").reset_index(drop=True)
cum_csv_path = os.path.join(output_report_dir, f"Cumulative_Feature_Selection_C{C_value}.csv")
df_cum.to_csv(cum_csv_path, index=False)
print(f"✅ Results successfully saved to: {cum_csv_path}")

# -----------------------------
# 4. Plot and Save Performance Curve
# -----------------------------
print(f"\nPlotting curve...")
plt.figure(figsize=(10, 6), dpi=300)

x = df_cum['Num_Features'].values
y_mean = df_cum['Mean_Accuracy(%)'].values
y_std = df_cum['Std_Dev(%)'].values

plt.plot(x, y_mean, label=f'Dual-Channel GLCM Features (C={C_value})', color='blue', linewidth=2)
if len(seeds) > 1:
    plt.fill_between(x, y_mean - y_std, y_mean + y_std, color='blue', alpha=0.15)

plt.title('Cumulative Feature Count vs. 5-Fold Cross-Validation Accuracy', fontsize=14, fontweight='bold')
plt.xlabel('Number of Top Features (Sorted by SHAP Importance)', fontsize=12)
plt.ylabel('Cross-Validation Accuracy (%)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11, loc='lower right')
plt.tight_layout()

plot_path = os.path.join(output_report_dir, f"Cumulative_Feature_Selection_Curve_C{C_value}.png")
plt.savefig(plot_path, dpi=300)
plt.show()

print(f"✅ Curve plot successfully saved to: {plot_path}")
print("\n🎉 Lightweight fast test completed successfully!")